In [0]:
%run ./utils

###Create config and utils

In [0]:
from pyspark.sql.functions import *

try:
    write_log("BRONZE_START", "STARTED", "Bronze ingestion started")

    # Get latest 2 files from bronze folder
    path_new, path_old = get_latest_files()

    write_log("BRONZE_FILES", "INFO", f"New: {path_new}, Old: {path_old}")

    df_new=read_dynamic(path_new)

    df_old=read_dynamic(path_old)

    # Add ingestion timestamp
    df_new=df_new.withColumn("ingestion_time", current_timestamp())
    df_old=df_old.withColumn("ingestion_time", current_timestamp())

    # 80% Column Similarity Check
    c1, c2=set(df_new.columns), set(df_old.columns)
    similarity=len(c1 & c2) / len(c1 | c2)

    if similarity < 0.8:
        write_log("BRONZE_VALIDATION", "WARNING", f"Low column similarity: {similarity:.2%}")
    else:
        write_log("BRONZE_VALIDATION", "SUCCESS", f"Schema similarity OK: {similarity:.2%}")


    df_new.write.mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("amazon_project.bronze.current_raw")

    df_old.write.mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("amazon_project.bronze.previous_raw")

    write_log("BRONZE_END", "SUCCESS", "Files successfully ingested into Bronze tables")

except Exception as e:
    write_log("BRONZE_ERROR", "FAILED", str(e))
    raise